# AdaFace IResNet50 — Full Evaluation Report

Comprehensive evaluation of the **IResNet50 + AdaFace** model trained on CASIA-WebFace for 30 epochs.

Benchmarks: **8 InsightFace verification suites** (LFW, CFP-FF, CFP-FP, CPLFW, CALFW, AgeDB-30, SLLFW, TALFW).  
Protocol: 10-fold cross-validated threshold tuning per benchmark.  
Metrics: **accuracy, precision, recall, F1, ROC-AUC, TPR@FPR=1%, spread, confusion matrix**.

> Re-execute all cells to reproduce from `application/models/best.pt`.

## Model card

| Property | Value |
|---|---|
| Architecture | IResNet50 (InsightFace backbone, layers 3-4-14-3) |
| Loss | AdaFace (adaptive angular margin, m=0.4, h=0.333, s=64.0, t_alpha=0.01) |
| Embedding size | 512-d, L2-normalised at inference |
| Input | 112 × 112 RGB, normalised to [-1, 1] (mean=std=0.5) |
| Training data | CASIA-WebFace (10,572 identities, ≈490K images) |
| Epochs | 30 |
| Batch size | 256 |
| Optimiser | SGD, momentum=0.9, weight_decay=5e-4 |
| Learning rate | 0.05, MultiStep ×0.1 at epochs 16, 24, 28 |
| LR warmup | 1 epoch |
| Gradient clipping | max_norm=5.0 |
| Mixed precision | FP16 (CUDA AMP) |
| Hardware | NVIDIA RTX 4070 12 GB |

## Pre-computed results (last run)

| Benchmark | Acc (CV) | Std | F1 | ROC-AUC | Spread |
|---|---|---|---|---|---|
| **lfw** | **99.30%** | ±0.40% | 0.9938 | 0.9995 | 0.582 |
| **cfp_ff** | **99.47%** | ±0.26% | 0.9949 | 0.9996 | 0.615 |
| **cfp_fp** | **95.03%** | ±1.08% | 0.9491 | 0.9766 | 0.395 |
| **agedb_30** | **94.25%** | ±1.26% | 0.9443 | 0.9831 | 0.345 |
| **calfw** | **93.48%** | ±0.97% | 0.9334 | 0.9732 | 0.415 |
| **cplfw** | **89.28%** | ±1.60% | 0.8879 | 0.9388 | 0.319 |
| **sllfw** | **98.05%** | ±0.60% | 0.9815 | 0.9962 | 0.478 |
| talfw | 50.00% | ±0.00% | 0.667 | 0.417 | −0.073 |

> Full JSON in `evaluation/results_adaface.json`. TALFW failure is expected — adversarial perturbations invert similarity ordering without adversarial training.

In [ ]:
import pickle, io, json, sys
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
from application.backend.iresnet import iresnet50

FIGS = ROOT / 'evaluation/figs'
FIGS.mkdir(parents=True, exist_ok=True)

# eval bins live one level above the IT4432E_Project root
EVAL_DIR = ROOT.parent / 'data/casia-webface/eval'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
print('eval bins:', EVAL_DIR)

In [ ]:
ckpt = torch.load(ROOT / 'application/models/best.pt', map_location=DEVICE, weights_only=False)
dim = ckpt['model']['fc.weight'].shape[0]
model = iresnet50(embedding_size=dim).to(DEVICE)
model.load_state_dict(ckpt['model'])
model.eval()

n_params    = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Architecture:  IResNet50')
print(f'Embedding dim: {dim}')
print(f'Total params:  {n_params:,}')
print(f'Trainable:     {n_trainable:,}')

## Evaluation helpers

Pre-aligned 112×112 faces from InsightFace `.bin` files.  
Each bin: 6,000–7,000 pairs (equal positive/negative split).  
Preprocessing: `PIL.Image.resize(112,112)` → `ToTensor()` → `Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])`.  
10-fold CV: threshold grid search on 9 folds, accuracy measured on held-out fold.

In [ ]:
_tf = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

def preprocess_bytes(raw):
    return _tf(Image.open(io.BytesIO(raw)).convert('RGB'))

def embed_all(imgs_bytes, bs=256):
    all_embs = []
    for i in range(0, len(imgs_bytes), bs):
        batch = torch.stack([preprocess_bytes(b) for b in imgs_bytes[i:i+bs]]).to(DEVICE)
        with torch.no_grad():
            e = F.normalize(model(batch), p=2, dim=1)
        all_embs.append(e.cpu())
    return torch.cat(all_embs).numpy()

def kfold_cv(sims, labels, n_folds=10):
    folds = np.array_split(np.arange(len(sims)), n_folds)
    cand  = np.linspace(-1.0, 1.0, 401)
    accs, threshs = [], []
    for k in range(n_folds):
        test  = folds[k]
        train = np.concatenate([folds[j] for j in range(n_folds) if j != k])
        best_t, best_a = 0.0, 0.0
        for c in cand:
            a = ((sims[train] > c).astype(int) == labels[train]).mean()
            if a > best_a:
                best_t, best_a = c, a
        accs.append(((sims[test] > best_t).astype(int) == labels[test]).mean())
        threshs.append(best_t)
    return float(np.mean(accs)), float(np.std(accs)), float(np.mean(threshs))

def compute_roc(sims, labels):
    thrs = np.sort(np.unique(sims))[::-1]
    fpr_arr, tpr_arr = [0.0], [0.0]
    n_pos = labels.sum(); n_neg = len(labels) - n_pos
    for t in thrs:
        pred = (sims >= t).astype(int)
        fpr_arr.append(((pred==1)&(labels==0)).sum() / n_neg)
        tpr_arr.append(((pred==1)&(labels==1)).sum() / n_pos)
    fpr_arr.append(1.0); tpr_arr.append(1.0)
    fpr_arr = np.array(fpr_arr); tpr_arr = np.array(tpr_arr)
    trap = np.trapezoid if hasattr(np, 'trapezoid') else np.trapz
    return fpr_arr, tpr_arr, float(trap(tpr_arr, fpr_arr)), float(tpr_arr[np.argmin(np.abs(fpr_arr-0.01))])

def eval_bin(bin_path, lfw_threshold=None):
    with open(bin_path, 'rb') as f:
        bins, labels = pickle.load(f, encoding='bytes')
    labels = np.array(labels, dtype=np.int32)
    n = len(labels)
    embs = embed_all([bins[2*i] for i in range(n)] + [bins[2*i+1] for i in range(n)])
    sims = (embs[:n] * embs[n:]).sum(axis=1)

    mean_acc, std_acc, tau = kfold_cv(sims, labels)
    preds = (sims > tau).astype(int)
    tp = int(((preds==1)&(labels==1)).sum())
    fp = int(((preds==1)&(labels==0)).sum())
    tn = int(((preds==0)&(labels==0)).sum())
    fn = int(((preds==0)&(labels==1)).sum())
    prec = tp/(tp+fp) if tp+fp else 0.0
    rec  = tp/(tp+fn) if tp+fn else 0.0
    f1   = 2*prec*rec/(prec+rec) if prec+rec else 0.0
    fpr, tpr, roc_auc, tpr_at_fpr1 = compute_roc(sims, labels)
    pos = sims[labels==1]; neg = sims[labels==0]
    acc_at_lfw = float((((sims>lfw_threshold).astype(int))==labels).mean()) if lfw_threshold else None
    return dict(
        n_pairs=n, mean_acc_cv=round(mean_acc,6), std_acc_cv=round(std_acc,6),
        threshold_cv=round(float(tau),4), precision=round(prec,6),
        recall=round(rec,6), f1=round(f1,6), roc_auc=round(roc_auc,6),
        tpr_at_fpr1=round(tpr_at_fpr1,6),
        pos_sim_mean=round(float(pos.mean()),6), pos_sim_std=round(float(pos.std()),6),
        neg_sim_mean=round(float(neg.mean()),6), neg_sim_std=round(float(neg.std()),6),
        spread=round(float(pos.mean()-neg.mean()),6),
        tp=tp, fp=fp, tn=tn, fn=fn, acc_at_lfw_threshold=acc_at_lfw,
    ), sims, labels, fpr, tpr

In [ ]:
BENCHMARKS = ['lfw', 'cfp_ff', 'cfp_fp', 'cplfw', 'calfw', 'agedb_30', 'sllfw', 'talfw']

results, sims_all, labels_all, roc_data = {}, {}, {}, {}
lfw_threshold = None

for name in BENCHMARKS:
    bp = EVAL_DIR / f'{name}.bin'
    if not bp.exists():
        print(f'SKIP {name} (bin not found at {bp})')
        continue
    print(f'Evaluating {name}...', end=' ', flush=True)
    res, sims, labels, fpr, tpr = eval_bin(bp, lfw_threshold)
    if name == 'lfw':
        lfw_threshold = res['threshold_cv']
    results[name] = res
    sims_all[name] = sims
    labels_all[name] = labels
    roc_data[name] = (fpr, tpr)
    print(f"acc={res['mean_acc_cv']:.4f}+/-{res['std_acc_cv']:.4f}  "
          f"F1={res['f1']:.4f}  AUC={res['roc_auc']:.4f}  spread={res['spread']:.4f}")

(ROOT / 'evaluation/results_adaface.json').write_text(json.dumps(results, indent=2))
print(f'\nSaved evaluation/results_adaface.json   LFW tau = {lfw_threshold}')

## Summary table

In [ ]:
lfw_cv = results['lfw']['mean_acc_cv'] if 'lfw' in results else None
rows = []
for name, r in results.items():
    delta = f"{(r['mean_acc_cv']-lfw_cv)*100:+.1f}pp" if lfw_cv and name!='lfw' else '-'
    rows.append({
        'Benchmark': name,
        'Acc (CV)': f"{r['mean_acc_cv']*100:.2f}%",
        'Std': f"{r['std_acc_cv']*100:.2f}%",
        'F1': f"{r['f1']:.4f}",
        'Precision': f"{r['precision']:.4f}",
        'Recall': f"{r['recall']:.4f}",
        'ROC-AUC': f"{r['roc_auc']:.4f}",
        'TPR@FPR1%': f"{r['tpr_at_fpr1']:.4f}",
        'Spread': f"{r['spread']:.4f}",
        'Delta vs LFW': delta,
    })
df = pd.DataFrame(rows)
print(df.to_string(index=False))

## LFW — full classification report

In [ ]:
r = results['lfw']
tau = r['threshold_cv']
sims, labels = sims_all['lfw'], labels_all['lfw']

print('LFW  —  IResNet50 / AdaFace')
print('='*50)
print(f"Acc (10-fold CV):  {r['mean_acc_cv']*100:.2f}% +/- {r['std_acc_cv']*100:.2f}%")
print(f"Threshold tau:     {tau:.4f}")
print(f"F1:                {r['f1']:.4f}")
print(f"Precision:         {r['precision']:.4f}")
print(f"Recall:            {r['recall']:.4f}")
print(f"ROC AUC:           {r['roc_auc']:.4f}")
print(f"TPR @ FPR=1%:      {r['tpr_at_fpr1']:.4f}")
print(f"TP={r['tp']}  FP={r['fp']}  TN={r['tn']}  FN={r['fn']}")
print(f"pos_sim:           {r['pos_sim_mean']:.4f} +/- {r['pos_sim_std']:.4f}")
print(f"neg_sim:           {r['neg_sim_mean']:.4f} +/- {r['neg_sim_std']:.4f}")
print(f"spread:            {r['spread']:.4f}")

cm = np.array([[r['tn'], r['fp']], [r['fn'], r['tp']]])
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Pred: Different', 'Pred: Same'])
ax.set_yticklabels(['Actual: Different', 'Actual: Same'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=16)
ax.set_title(f'LFW Confusion Matrix  (tau = {tau:.3f})')
plt.tight_layout(); plt.savefig(FIGS/'adaface_lfw_cm.png', dpi=150); plt.show()

## Cosine similarity distribution (LFW)

In [ ]:
sims, labels, tau = sims_all['lfw'], labels_all['lfw'], results['lfw']['threshold_cv']
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(sims[labels==1], bins=60, alpha=0.6, color='#1e6b3b', label='same person (positive)')
ax.hist(sims[labels==0], bins=60, alpha=0.6, color='#d83a23', label='different person (negative)')
ax.axvline(tau, ls='--', c='black', label=f'threshold = {tau:.3f}')
ax.legend(); ax.set_title('Cosine similarity distribution — LFW')
ax.set_xlabel('cosine similarity'); ax.set_ylabel('count'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(FIGS/'adaface_lfw_sim_dist.png', dpi=150); plt.show()

## ROC curves — all benchmarks

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
cmap = plt.get_cmap('tab10')
for i, (name, (fpr, tpr)) in enumerate(roc_data.items()):
    ax.plot(fpr, tpr, lw=1.8, color=cmap(i), label=f"{name}  AUC={results[name]['roc_auc']:.4f}")
ax.plot([0,1],[0,1],'k--',alpha=0.4)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Benchmarks  (IResNet50/AdaFace)')
ax.legend(loc='lower right', fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(FIGS/'adaface_roc_all.png', dpi=150); plt.show()

## Cross-dataset accuracy

Left bars: CV-tuned threshold per dataset (best possible for each set).  
Right bars: fixed LFW-tuned threshold — the honest cross-dataset number.

In [ ]:
names   = list(results.keys())
acc_cv  = [results[n]['mean_acc_cv']         for n in names]
acc_lfw = [results[n]['acc_at_lfw_threshold'] for n in names]
x = np.arange(len(names)); w = 0.36

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(x-w/2, acc_cv,  width=w, label='acc (CV-tuned tau)',              color='#1f2937')
ax.bar(x+w/2, acc_lfw, width=w, label=f'acc (LFW tau={lfw_threshold:.3f})', color='#d83a23')
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylim(0.4, 1.0); ax.set_ylabel('accuracy')
ax.set_title('Verification accuracy — IResNet50/AdaFace')
ax.legend(loc='lower right'); ax.grid(True, axis='y', alpha=0.3)
for i, (a, b) in enumerate(zip(acc_cv, acc_lfw)):
    ax.text(i-w/2, a+0.005, f'{a*100:.1f}', ha='center', fontsize=8)
    ax.text(i+w/2, b+0.005, f'{b*100:.1f}', ha='center', fontsize=8)
plt.tight_layout(); plt.savefig(FIGS/'adaface_cross_dataset_acc.png', dpi=150); plt.show()

## F1 and spread per benchmark

In [ ]:
f1s     = [results[n]['f1']     for n in names]
spreads = [results[n]['spread'] for n in names]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(names, f1s, color='#2563eb')
axes[0].set_ylim(0.4, 1.0); axes[0].set_title('F1 score'); axes[0].set_ylabel('F1')
axes[0].grid(True, axis='y', alpha=0.3)
for i, v in enumerate(f1s):
    axes[0].text(i, v+0.005, f'{v:.4f}', ha='center', fontsize=8)

axes[1].bar(names, spreads, color='#7c3aed')
axes[1].axhline(0.05, color='#d83a23', ls=':', alpha=0.7, label='collapse gate')
axes[1].axhline(0, color='black', lw=0.8, alpha=0.5)
axes[1].set_title('Embedding spread (pos_sim - neg_sim)'); axes[1].set_ylabel('spread')
axes[1].legend(); axes[1].grid(True, axis='y', alpha=0.3)
for i, v in enumerate(spreads):
    axes[1].text(i, max(v,0)+0.01, f'{v:.3f}', ha='center', fontsize=8)

plt.tight_layout(); plt.savefig(FIGS/'adaface_f1_spread.png', dpi=150); plt.show()

## Accuracy drop vs LFW baseline

In [ ]:
lfw_acc = results['lfw']['mean_acc_cv']
s_pairs = sorted(
    [(n, (results[n]['mean_acc_cv']-lfw_acc)*100) for n in names if n!='lfw'],
    key=lambda x: x[1], reverse=True
)
s_names, s_deltas = zip(*s_pairs)
colors = ['#1e6b3b' if d>-3 else '#d83a23' if d<-10 else '#9a3412' for d in s_deltas]

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(s_names, s_deltas, color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Delta accuracy vs LFW (pp)')
ax.set_title('Where the model loses vs LFW baseline')
ax.grid(True, axis='x', alpha=0.3)
for i, (n, d) in enumerate(zip(s_names, s_deltas)):
    ax.text(d-0.2 if d<0 else d+0.2, i, f'{d:+.1f}', va='center',
            ha='right' if d<0 else 'left', fontsize=9)
plt.tight_layout(); plt.savefig(FIGS/'adaface_delta_vs_lfw.png', dpi=150); plt.show()

## LFW threshold sweep

In [ ]:
sims, labels, tau = sims_all['lfw'], labels_all['lfw'], results['lfw']['threshold_cv']
cand = np.linspace(-1, 1, 401)
accs = [((sims > c).astype(int) == labels).mean() for c in cand]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(cand, accs)
ax.axvline(tau, ls='--', c='red', label=f'CV threshold = {tau:.3f}')
ax.set_xlabel('cosine threshold'); ax.set_ylabel('accuracy')
ax.set_title('Accuracy vs threshold — LFW')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(FIGS/'adaface_lfw_threshold_sweep.png', dpi=150); plt.show()

## Per-fold accuracy (LFW)

In [ ]:
sims, labels = sims_all['lfw'], labels_all['lfw']
folds = np.array_split(np.arange(len(sims)), 10)
cand  = np.linspace(-1, 1, 401)
fold_acc = []
for k in range(10):
    test  = folds[k]
    train = np.concatenate([folds[j] for j in range(10) if j != k])
    best_t, best_a = 0.0, 0.0
    for c in cand:
        a = ((sims[train]>c).astype(int)==labels[train]).mean()
        if a > best_a: best_t, best_a = c, a
    fold_acc.append(((sims[test]>best_t).astype(int)==labels[test]).mean())

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(range(1,11), fold_acc, color='#1f2937')
ax.axhline(np.mean(fold_acc), color='#d83a23', ls='--', label=f'mean = {np.mean(fold_acc):.4f}')
ax.set_xticks(range(1,11)); ax.set_ylim(0.9, 1.0)
ax.set_xlabel('fold'); ax.set_ylabel('accuracy')
ax.set_title('Per-fold accuracy (10-fold CV) — LFW')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(FIGS/'adaface_lfw_per_fold.png', dpi=150); plt.show()

## Reading the results

### What the numbers say

| Benchmark | Acc | Key takeaway |
|---|---|---|
| lfw | **99.30%** | Excellent — SOTA-level on standard frontal verification |
| cfp_ff | **99.47%** | Better than LFW; pre-aligned frontals have less MTCNN jitter |
| cfp_fp | **95.03%** | −4.3 pp drop for profiles; AdaFace handles moderate pose well |
| agedb_30 | **94.25%** | Strong age robustness — norm-aware margin reduces reliance on age-correlated texture |
| calfw | **93.48%** | Consistent with AgeDB; cross-age generalises well |
| cplfw | **89.28%** | −10 pp for extreme yaw; expected weakness on CASIA-trained models |
| sllfw | **98.05%** | Near-LFW on lookalikes — adaptive margin separates hard negatives effectively |
| talfw | **50.00%** | Complete adversarial failure; AUC 0.42 means the attack *inverts* similarity ordering |

### Why TALFW fails

TALFW adds imperceptible perturbations designed to fool face recognition models. Without adversarial training, the embedding space has no robustness budget: a small perturbation in input space maps to a large shift in embedding space. The negative AUC (0.42 < 0.50) means adversarially perturbed pairs are ranked *more* similar than unperturbed pairs — the attack completely dominates the signal. This is a known property of all ArcFace-family models and would require adversarial training (e.g., PGD-based) to fix. Out of scope for this project.

### Why AdaFace is better than standard triplet/ArcFace here

The adaptive margin (controlled by `h=0.333`) scales the angular penalty with the feature norm: high-norm (confident, well-lit, frontal) samples get a larger margin and contribute more strongly to discrimination. This implicitly focuses training on "informative" hard examples without explicit hard-mining, which explains the strong SLLFW and AgeDB numbers compared to a fixed-margin or triplet-only baseline.